In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
import pandas as pd
import os
import zipfile
from google.colab import files

# ==========================================
# 1. Physical Layer: SDE & First Passage Time
# ==========================================
def calc_fpt_linear_cdf(mu, sigma, L, dt):
    """
    Calculates the theoretical baseline probability of cognitive degeneracy (skip)
    using the Inverse Gaussian CDF derived from the Linear Drift SDE.
    """
    if mu <= 0 or sigma <= 0 or dt <= 0:
        return 0.0

    term1 = (mu * dt - L) / (sigma * np.sqrt(dt))
    term2 = (-mu * dt - L) / (sigma * np.sqrt(dt))

    # Analytical solution for First Passage Time CDF
    prob = norm.cdf(term1) + np.exp(2 * mu * L / (sigma**2)) * norm.cdf(term2)
    return np.clip(prob, 0.0, 1.0)

# ==========================================
# 2. Observation Layer: Structural Coupling & Noise
# ==========================================
def apply_beta_noise(p_base, K, size):
    """
    Applies assessor bias and local observation uncertainty via Beta distribution.
    Structurally coupled to the physical baseline p_base.
    """
    if p_base <= 0.0:
        return np.zeros(size)
    if p_base >= 1.0:
        return np.ones(size)

    alpha = p_base * K
    beta_param = (1.0 - p_base) * K
    return np.random.beta(alpha, beta_param, size)

# ==========================================
# 3. Macroscopic Layer: Monte Carlo Phase Space Exploration
# ==========================================
def explore_phase_space():
    # 結果保存用のディレクトリ作成
    output_dir = "simulation_results"
    os.makedirs(output_dir, exist_ok=True)

    # Dimensionless Constants
    X_A, X_C = 0.0, 0.85
    L_AC = X_C - X_A  # Distance for A -> C Skip Degeneracy

    # Simulation Parameters
    mu = 0.1          # Constant drift rate
    K = 100           # Noise concentration (reliability of observation)
    N = 10000         # Number of Monte Carlo trials per grid point

    # Grid for Phase Space (X: Observation Interval, Y: Environmental Fluctuation)
    dt_vals = np.linspace(0.01, 1.0, 40)
    sigma_vals = np.linspace(0.01, 0.5, 40)

    # Results matrix
    skip_rates = np.zeros((len(sigma_vals), len(dt_vals)))

    print("Running Monte Carlo Phase Space Exploration...")

    for i, sig in enumerate(sigma_vals):
        for j, dt in enumerate(dt_vals):
            # Step A: Derive physical baseline probability
            p_AC_base = calc_fpt_linear_cdf(mu, sig, L_AC, dt)

            # Step B: Superimpose observation noise
            p_AC_eff = apply_beta_noise(p_AC_base, K, N)

            # Step C: Macroscopic Monte Carlo evaluation
            skips = np.sum(np.random.rand(N) < p_AC_eff)
            skip_rates[i, j] = skips / N

    # ==========================================
    # 4. Export CSV
    # ==========================================
    df = pd.DataFrame(skip_rates, index=np.round(sigma_vals, 3), columns=np.round(dt_vals, 3))
    df.index.name = "Sigma"
    df.columns.name = "dt"
    csv_path = os.path.join(output_dir, "skip_rates_matrix.csv")
    df.to_csv(csv_path)
    print(f"Saved CSV: {csv_path}")

    # ==========================================
    # 5. Visualization (Multiple Graphs)
    # ==========================================
    # グラフ1: ヒートマップ (Heatmap)
    plt.figure(figsize=(10, 8))
    ax1 = sns.heatmap(skip_rates,
                     xticklabels=np.round(dt_vals, 2),
                     yticklabels=np.round(sigma_vals, 2),
                     cmap="viridis", cbar_kws={'label': 'Skip Degeneracy Rate (A->C)'})
    ax1.invert_yaxis()
    plt.title("Phase Space of Cognitive Degeneracy\nLinear Drift SDE vs Discrete Observation", fontsize=14, pad=15)
    plt.xlabel(r"Observation Interval ($\Delta t$)", fontsize=12)
    plt.ylabel(r"Environmental Fluctuation ($\sigma$)", fontsize=12)
    ax1.set_xticks(ax1.get_xticks()[::4])
    ax1.set_yticks(ax1.get_yticks()[::4])
    plt.tight_layout()
    heatmap_path = os.path.join(output_dir, "fig1_heatmap.png")
    plt.savefig(heatmap_path, dpi=300)
    plt.close()

    # グラフ2: 断面折れ線グラフ (Cross-section Line Plot)
    plt.figure(figsize=(10, 6))
    # 異なるゆらぎ(sigma)の代表値を抽出して描画
    indices_to_plot = [5, 20, 35]
    for idx in indices_to_plot:
        plt.plot(dt_vals, skip_rates[idx, :], marker='o', markersize=4,
                 label=fr"$\sigma$ = {sigma_vals[idx]:.2f}")
    plt.title("Skip Degeneracy Rate vs Observation Interval", fontsize=14)
    plt.xlabel(r"Observation Interval ($\Delta t$)", fontsize=12)
    plt.ylabel("Skip Degeneracy Rate (A->C)", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    lineplot_path = os.path.join(output_dir, "fig2_lineplot.png")
    plt.savefig(lineplot_path, dpi=300)
    plt.close()

    # グラフ3: 3Dサーフェスプロット (3D Surface Plot)
    fig = plt.figure(figsize=(12, 8))
    ax3d = fig.add_subplot(111, projection='3d')
    DT, SIG = np.meshgrid(dt_vals, sigma_vals)
    surf = ax3d.plot_surface(DT, SIG, skip_rates, cmap='viridis', edgecolor='none', alpha=0.9)
    ax3d.set_title("3D Surface of Phase Space", fontsize=14, pad=15)
    ax3d.set_xlabel(r"$\Delta t$", fontsize=12)
    ax3d.set_ylabel(r"$\sigma$", fontsize=12)
    ax3d.set_zlabel("Skip Rate", fontsize=12)
    fig.colorbar(surf, ax=ax3d, shrink=0.5, aspect=10, label='Skip Degeneracy Rate')
    surface_path = os.path.join(output_dir, "fig3_surface.png")
    plt.savefig(surface_path, dpi=300)
    plt.close()

    print("Graphs generated and saved.")

    # ==========================================
    # 6. Zip and Download
    # ==========================================
    zip_filename = "cognitive_degeneracy_results.zip"
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files_in_dir in os.walk(output_dir):
            for file in files_in_dir:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.basename(file_path))

    print(f"Results zipped successfully into {zip_filename}")

    # Colabの自動ダウンロードトリガー
    try:
        files.download(zip_filename)
        print("Download triggered automatically.")
    except Exception as e:
        print(f"Error triggering download (Ensure you are running in Google Colab): {e}")

if __name__ == "__main__":
    explore_phase_space()

Running Monte Carlo Phase Space Exploration...


/tmp/ipykernel_1641/3068340940.py:25: RuntimeWarning: overflow encountered in exp
  prob = norm.cdf(term1) + np.exp(2 * mu * L / (sigma**2)) * norm.cdf(term2)
/tmp/ipykernel_1641/3068340940.py:25: RuntimeWarning: invalid value encountered in scalar multiply
  prob = norm.cdf(term1) + np.exp(2 * mu * L / (sigma**2)) * norm.cdf(term2)


Saved CSV: simulation_results/skip_rates_matrix.csv
Graphs generated and saved.
Results zipped successfully into cognitive_degeneracy_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered automatically.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import zipfile
from google.colab import files

# ==========================================
# 1. Physical Layer: Non-linear SDE Paths
# ==========================================
def simulate_nonlinear_sde_skips(gamma, dt_obs, sigma=0.1, k=0.1, N_paths=2000):
    """
    Simulates physical deterioration paths using Euler-Maruyama integration
    for the non-linear SDE: dX = k * t^gamma * dt + sigma * dW.
    Returns the baseline probability of A->C cognitive skip.
    """
    X_A, X_B, X_C = 0.0, 0.60, 0.85

    # Simulation Time Setup
    T_max = 12.0
    dt_sim = 0.01  # Microscopic physical time step
    n_steps = int(T_max / dt_sim)

    # Time array and Drift calculation
    t_array = np.linspace(0, T_max, n_steps)
    mu_array = k * (t_array ** gamma)

    # Wiener process (Brownian motion)
    dW = np.random.randn(N_paths, n_steps) * np.sqrt(dt_sim)

    # Vectorized SDE Integration
    dX = mu_array * dt_sim + sigma * dW
    X = np.cumsum(dX, axis=1)

    # Discrete Observation Sampling
    step_size = max(1, int(np.round(dt_obs / dt_sim)))
    obs_X = X[:, ::step_size]

    # Detect Cognitive Degeneracy (Skip A -> C between consecutive observations)
    # State A: X < X_B, State C: X >= X_C
    is_A = obs_X[:, :-1] < X_B
    is_C = obs_X[:, 1:] >= X_C
    skips = is_A & is_C

    # Calculate base probability of an asset experiencing at least one A->C skip
    p_base = np.mean(np.any(skips, axis=1))
    return p_base

# ==========================================
# 2. Observation Layer: Structural Coupling
# ==========================================
def apply_beta_noise(p_base, K, size):
    if p_base <= 0.0: return np.zeros(size)
    if p_base >= 1.0: return np.ones(size)
    alpha = p_base * K
    beta_param = (1.0 - p_base) * K
    return np.random.beta(alpha, beta_param, size)

# ==========================================
# 3. Macroscopic Phase Space Engine
# ==========================================
def explore_nonlinear_phase_space():
    output_dir = "nonlinear_results"
    os.makedirs(output_dir, exist_ok=True)

    # Variables for Phase Space
    dt_vals = np.linspace(0.1, 1.5, 30)       # X-axis: Observation Interval
    gamma_vals = np.linspace(0.5, 3.0, 30)    # Y-axis: Acceleration Parameter

    K_noise = 100   # Assessor bias variance
    N_macro = 5000  # Cohort size for macroscopic evaluation

    skip_rates = np.zeros((len(gamma_vals), len(dt_vals)))

    print("Running Non-Linear Monte Carlo Phase Space Exploration...")
    print("This may take a minute due to SDE path integration...")

    for i, gamma in enumerate(gamma_vals):
        for j, dt in enumerate(dt_vals):
            # Step A: Derive baseline skip probability from physical SDE paths
            p_base = simulate_nonlinear_sde_skips(gamma, dt)

            # Step B: Apply structural beta noise
            p_eff = apply_beta_noise(p_base, K_noise, N_macro)

            # Step C: Macroscopic evaluation
            skip_rates[i, j] = np.sum(np.random.rand(N_macro) < p_eff) / N_macro

    # ==========================================
    # 4. Data Export & Visualization
    # ==========================================
    # Save CSV
    df = pd.DataFrame(skip_rates, index=np.round(gamma_vals, 3), columns=np.round(dt_vals, 3))
    df.index.name = "Gamma"
    df.columns.name = "dt"
    csv_path = os.path.join(output_dir, "nonlinear_skip_matrix.csv")
    df.to_csv(csv_path)

    # Figure 1: Heatmap
    plt.figure(figsize=(10, 8))
    ax1 = sns.heatmap(skip_rates,
                     xticklabels=np.round(dt_vals, 2),
                     yticklabels=np.round(gamma_vals, 2),
                     cmap="magma", cbar_kws={'label': 'Skip Degeneracy Rate (A->C)'})
    ax1.invert_yaxis()
    plt.title(r"Phase Space of Cognitive Degeneracy vs Acceleration ($\gamma$)", fontsize=14, pad=15)
    plt.xlabel(r"Observation Interval ($\Delta t$)", fontsize=12)
    plt.ylabel(r"Acceleration Parameter ($\gamma$)", fontsize=12)
    ax1.set_xticks(ax1.get_xticks()[::3])
    ax1.set_yticks(ax1.get_yticks()[::3])
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig1_nonlinear_heatmap.png"), dpi=300)
    plt.close()

    # Figure 2: Cross-section Line Plot
    plt.figure(figsize=(10, 6))
    indices = [5, 15, 25]  # Select different gamma levels
    for idx in indices:
        plt.plot(dt_vals, skip_rates[idx, :], marker='o', markersize=4,
                 label=fr"$\gamma$ = {gamma_vals[idx]:.2f}")
    plt.axhline(0.2, color='red', linestyle='--', alpha=0.6, label="Tolerance Threshold (20%)")
    plt.title(r"Collapse of Homogenität: Skip Rates by $\gamma$", fontsize=14)
    plt.xlabel(r"Observation Interval ($\Delta t$)", fontsize=12)
    plt.ylabel("Skip Degeneracy Rate (A->C)", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig2_nonlinear_lineplot.png"), dpi=300)
    plt.close()

    # Figure 3: 3D Surface
    fig = plt.figure(figsize=(12, 8))
    ax3d = fig.add_subplot(111, projection='3d')
    DT, GAM = np.meshgrid(dt_vals, gamma_vals)
    surf = ax3d.plot_surface(DT, GAM, skip_rates, cmap='magma', edgecolor='none', alpha=0.9)
    ax3d.set_title("3D Boundary of Statistical Homogeneity", fontsize=14)
    ax3d.set_xlabel(r"$\Delta t$", fontsize=12)
    ax3d.set_ylabel(r"$\gamma$", fontsize=12)
    ax3d.set_zlabel("Skip Rate", fontsize=12)
    fig.colorbar(surf, ax=ax3d, shrink=0.5, aspect=10)
    plt.savefig(os.path.join(output_dir, "fig3_nonlinear_surface.png"), dpi=300)
    plt.close()

    print("Graphs generated and saved.")

    # ==========================================
    # 5. Zip and Download
    # ==========================================
    zip_filename = "nonlinear_degeneracy_results.zip"
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files_in_dir in os.walk(output_dir):
            for file in files_in_dir:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.basename(file_path))

    print(f"Results zipped successfully into {zip_filename}")

    try:
        files.download(zip_filename)
        print("Download triggered automatically.")
    except Exception as e:
        print(f"Error triggering download: {e}")

if __name__ == "__main__":
    explore_nonlinear_phase_space()

Running Non-Linear Monte Carlo Phase Space Exploration...
This may take a minute due to SDE path integration...
Graphs generated and saved.
Results zipped successfully into nonlinear_degeneracy_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered automatically.


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import zipfile
from google.colab import files

# ==========================================
# 1. Accelerated Markov Engine (Vectorized)
# ==========================================
def simulate_cohort(sigma_AB, sigma_BC, gamma=2.0, N=100000):
    """
    Simulates a cohort of N assets under non-linear accelerated deterioration.
    Vectorized for high-performance computing (10^5 to 10^6 scale).
    """
    T_max = 30.0
    dt = 0.5
    n_steps = int(T_max / dt)

    # Time-dependent base probabilities (Non-linear drift: mu(t) ~ t^gamma)
    t_array = np.linspace(dt, T_max, n_steps)
    k_AB, k_BC, k_CD = 0.05, 0.15, 0.30  # Physical speed constants

    p_base_AB = 1.0 - np.exp(-k_AB * (t_array**gamma) * dt)
    p_base_BC = 1.0 - np.exp(-k_BC * (t_array**gamma) * dt)
    p_base_CD = 1.0 - np.exp(-k_CD * (t_array**gamma) * dt)

    # State array: 0=A, 1=B, 2=C, 3=D
    states = np.zeros(N, dtype=np.int8)
    lifespans = np.full(N, T_max)
    skips_AC = np.zeros(N, dtype=bool)

    for step in range(n_steps):
        # 1. A -> B Transitions
        idx_A = (states == 0)
        n_A = np.sum(idx_A)
        if n_A > 0:
            # Apply targeted randomization (Gaussian noise with clamping)
            p_eff_AB = np.clip(p_base_AB[step] + np.random.normal(0, sigma_AB, n_A), 0, 1)
            trans_AB = np.random.rand(n_A) < p_eff_AB
            states[idx_A] += trans_AB

            # Immediately evaluate B->C for those who just transitioned (Cognitive Skip A->C)
            just_transitioned = np.zeros(N, dtype=bool)
            np.place(just_transitioned, idx_A, trans_AB)

            n_just = np.sum(just_transitioned)
            if n_just > 0:
                p_eff_BC_skip = np.clip(p_base_BC[step] + np.random.normal(0, sigma_BC, n_just), 0, 1)
                trans_skip = np.random.rand(n_just) < p_eff_BC_skip
                states[just_transitioned] += trans_skip

                # Record cognitive skips
                skips_mapped = np.zeros(N, dtype=bool)
                np.place(skips_mapped, just_transitioned, trans_skip)
                skips_AC |= skips_mapped

        # 2. B -> C Transitions (Normal)
        idx_B = (states == 1) & ~skips_AC  # Exclude those who just skipped this step
        n_B = np.sum(idx_B)
        if n_B > 0:
            p_eff_BC = np.clip(p_base_BC[step] + np.random.normal(0, sigma_BC, n_B), 0, 1)
            states[idx_B] += (np.random.rand(n_B) < p_eff_BC)

        # 3. C -> D Transitions
        idx_C = (states == 2)
        n_C = np.sum(idx_C)
        if n_C > 0:
            p_eff_CD = np.clip(p_base_CD[step], 0, 1) # CD is assumed deterministic/stable
            trans_CD = np.random.rand(n_C) < p_eff_CD
            states[idx_C] += trans_CD

            # Record lifespan for those who reached D in this step
            reached_D = np.zeros(N, dtype=bool)
            np.place(reached_D, idx_C, trans_CD)
            lifespans[reached_D] = t_array[step]

    # Metrics for Homogeneity
    lifespan_std = np.std(lifespans)
    skip_rate = np.mean(skips_AC)

    return lifespan_std, skip_rate

# ==========================================
# 2. Phase Space Exploration
# ==========================================
def explore_randomization_strategies():
    output_dir = "randomization_threshold_results"
    os.makedirs(output_dir, exist_ok=True)

    # Grid search for Randomization parameters
    sigma_AB_vals = np.linspace(0.01, 0.30, 20)
    sigma_BC_vals = np.linspace(0.01, 0.30, 20)
    N_trials = 100000  # 100,000 per grid point

    std_matrix = np.zeros((len(sigma_BC_vals), len(sigma_AB_vals)))
    skip_matrix = np.zeros((len(sigma_BC_vals), len(sigma_AB_vals)))

    print(f"Running Phase Space Search: {len(sigma_AB_vals)*len(sigma_BC_vals)} grids x {N_trials} sims...")

    for i, sig_BC in enumerate(sigma_BC_vals):
        for j, sig_AB in enumerate(sigma_AB_vals):
            std_dev, skip = simulate_cohort(sig_AB, sig_BC, gamma=2.5, N=N_trials)
            std_matrix[i, j] = std_dev
            skip_matrix[i, j] = skip

    # ==========================================
    # 3. Data Export & Visualization
    # ==========================================
    # CSV
    df_std = pd.DataFrame(std_matrix, index=np.round(sigma_BC_vals, 3), columns=np.round(sigma_AB_vals, 3))
    df_std.to_csv(os.path.join(output_dir, "lifespan_std_matrix.csv"))

    # Fig 1: Heatmap of Lifespan StdDev (Stability of Homogenität)
    # Lower StdDev means higher stability/homogeneity
    plt.figure(figsize=(10, 8))
    ax1 = sns.heatmap(std_matrix, xticklabels=np.round(sigma_AB_vals, 2), yticklabels=np.round(sigma_BC_vals, 2),
                      cmap="coolwarm", cbar_kws={'label': 'Lifespan Std Dev (Years)'})
    ax1.invert_yaxis()
    plt.title("Stability of Homogenität via Transition Randomization\n(Lower Std Dev = Higher Stability)", fontsize=14, pad=15)
    plt.xlabel(r"Randomization of Initial Degradation ($\sigma_{A \to B}$)", fontsize=12)
    plt.ylabel(r"Randomization of Critical Transition ($\sigma_{B \to C}$)", fontsize=12)
    ax1.set_xticks(ax1.get_xticks()[::2])
    ax1.set_yticks(ax1.get_yticks()[::2])
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig1_homogeneity_stability.png"), dpi=300)
    plt.close()

    # Fig 2: 3D Surface of Skip Degeneracy
    fig = plt.figure(figsize=(12, 8))
    ax3d = fig.add_subplot(111, projection='3d')
    X_AB, Y_BC = np.meshgrid(sigma_AB_vals, sigma_BC_vals)
    surf = ax3d.plot_surface(X_AB, Y_BC, skip_matrix, cmap='magma', edgecolor='none', alpha=0.9)
    ax3d.set_title("3D Phase Space of Cognitive Skips vs Randomization", fontsize=14)
    ax3d.set_xlabel(r"$\sigma_{A \to B}$", fontsize=12)
    ax3d.set_ylabel(r"$\sigma_{B \to C}$", fontsize=12)
    ax3d.set_zlabel("Skip Rate (A->C)", fontsize=12)
    fig.colorbar(surf, ax=ax3d, shrink=0.5, aspect=10)
    plt.savefig(os.path.join(output_dir, "fig2_skip_surface.png"), dpi=300)
    plt.close()

    # Fig 3: Cross-sectional Comparison (Which one to randomize?)
    plt.figure(figsize=(10, 6))
    mid_idx = len(sigma_AB_vals) // 2
    plt.plot(sigma_AB_vals, std_matrix[0, :], 'b-o', label=r"Varying $\sigma_{A \to B}$ (with minimal $\sigma_{B \to C}$)")
    plt.plot(sigma_BC_vals, std_matrix[:, 0], 'r-x', label=r"Varying $\sigma_{B \to C}$ (with minimal $\sigma_{A \to B}$)")
    plt.title("Impact of Target Randomization on Lifespan Stability", fontsize=14)
    plt.xlabel("Magnitude of Randomization / Uncertainty", fontsize=12)
    plt.ylabel("Lifespan Standard Deviation (Years)", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig3_strategy_comparison.png"), dpi=300)
    plt.close()

    print("Simulations complete. Graphs generated.")

    # ==========================================
    # 4. Zip and Download
    # ==========================================
    zip_filename = "randomization_strategy_results.zip"
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files_in_dir in os.walk(output_dir):
            for file in files_in_dir:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.basename(file_path))

    files.download(zip_filename)

if __name__ == "__main__":
    explore_randomization_strategies()

Running Phase Space Search: 400 grids x 100000 sims...
Simulations complete. Graphs generated.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import zipfile
from google.colab import files

# ==========================================
# 1. Unified Markov-Jump Engine (Vectorized)
# ==========================================
def simulate_robust_jump_randomization(sigma_AB, lam, gamma=2.5, N=50000):
    """
    Simulates a cohort facing BOTH non-linear accelerated degradation AND
    sudden Poisson shocks (Jump-Diffusion equivalent).
    Evaluates if initial randomization (sigma_AB) can absorb the shocks.
    """
    T_max = 30.0
    dt = 0.5
    n_steps = int(T_max / dt)

    # Continuous Acceleration Drift
    t_array = np.linspace(dt, T_max, n_steps)
    k_AB, k_BC, k_CD = 0.05, 0.15, 0.30
    p_base_AB = 1.0 - np.exp(-k_AB * (t_array**gamma) * dt)
    p_base_BC = 1.0 - np.exp(-k_BC * (t_array**gamma) * dt)
    p_base_CD = 1.0 - np.exp(-k_CD * (t_array**gamma) * dt)

    # Poisson Jump Probability (Sudden Physical Shocks)
    p_jump = 1.0 - np.exp(-lam * dt)

    states = np.zeros(N, dtype=np.int8)
    lifespans = np.full(N, T_max)

    for step in range(n_steps):
        # 1. Evaluate Sudden Physical Jumps FIRST
        idx_A = (states == 0)
        idx_B = (states == 1)

        jumps_A = np.random.rand(np.sum(idx_A)) < p_jump
        states[idx_A] += jumps_A * 2  # A -> C Sudden Skip

        jumps_B = np.random.rand(np.sum(idx_B)) < p_jump
        states[idx_B] += jumps_B * 2  # B -> D Sudden Fatal Skip

        # Re-evaluate indices after sudden jumps
        idx_A = (states == 0)
        idx_B = (states == 1)
        idx_C = (states == 2)

        # 2. Targeted Randomization: Cognitive smoothing on A -> B ONLY
        n_A = np.sum(idx_A)
        if n_A > 0:
            # Apply Gaussian noise based on the discovered winning strategy
            p_eff_AB = np.clip(p_base_AB[step] + np.random.normal(0, sigma_AB, n_A), 0, 1)
            states[idx_A] += (np.random.rand(n_A) < p_eff_AB)

        # 3. Strict Determinism on B -> C and C -> D (As proven necessary)
        n_B = np.sum(idx_B)
        if n_B > 0:
            states[idx_B] += (np.random.rand(n_B) < p_base_BC[step])

        n_C = np.sum(idx_C)
        if n_C > 0:
            trans_CD = np.random.rand(n_C) < p_base_CD[step]
            states[idx_C] += trans_CD

            # Record lifespan
            reached_D = np.zeros(N, dtype=bool)
            np.place(reached_D, idx_C, trans_CD)
            lifespans[reached_D] = t_array[step]

    return np.std(lifespans)

# ==========================================
# 2. Phase Space Exploration
# ==========================================
def explore_effective_thresholds():
    output_dir = "robust_threshold_results"
    os.makedirs(output_dir, exist_ok=True)

    # Grid search: Sudden Shock Freq vs Randomization Magnitude
    lambda_vals = np.linspace(0.01, 0.20, 20)      # X-axis: Environmental Severity
    sigma_AB_vals = np.linspace(0.0, 0.40, 25)     # Y-axis: Cognitive Randomization
    N_trials = 50000

    std_matrix = np.zeros((len(sigma_AB_vals), len(lambda_vals)))

    print(f"Running Synthesis Phase Space Search: {len(lambda_vals)*len(sigma_AB_vals)} grids...")

    for i, sig_AB in enumerate(sigma_AB_vals):
        for j, lam in enumerate(lambda_vals):
            std_matrix[i, j] = simulate_robust_jump_randomization(sig_AB, lam, gamma=2.5, N=N_trials)

    # ==========================================
    # 3. Data Export & Visualization
    # ==========================================
    # CSV
    df_std = pd.DataFrame(std_matrix, index=np.round(sigma_AB_vals, 3), columns=np.round(lambda_vals, 3))
    df_std.index.name = "Sigma_AB"
    df_std.columns.name = "Lambda"
    df_std.to_csv(os.path.join(output_dir, "shock_absorption_matrix.csv"))

    # Fig 1: Heatmap (Finding the stable valley)
    plt.figure(figsize=(10, 8))
    ax1 = sns.heatmap(std_matrix, xticklabels=np.round(lambda_vals, 3), yticklabels=np.round(sigma_AB_vals, 2), cmap="viridis_r", cbar_kws={'label': 'Lifespan Std Dev (Lower = More Stable)'})
    ax1.invert_yaxis()
    plt.title("Effective Thresholds: Absorbing Jump Shocks via Randomization", fontsize=14, pad=15)
    plt.xlabel(r"Frequency of Sudden Shocks ($\lambda$)", fontsize=12)
    plt.ylabel(r"Initial Degradation Randomization ($\sigma_{AB}$)", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig1_effective_threshold_heatmap.png"), dpi=300)
    plt.close()

    # Fig 2: Cross-section (The "U-shape" optimization curve)
    plt.figure(figsize=(10, 6))
    indices = [5, 10, 15] # Different shock levels
    for idx in indices:
        plt.plot(sigma_AB_vals, std_matrix[:, idx], marker='o', label=fr"Shock $\lambda$ = {lambda_vals[idx]:.3f}")
    plt.title("Optimization Curve: Finding the Minimum Variance Threshold", fontsize=14)
    plt.xlabel(r"Magnitude of Randomization ($\sigma_{AB}$)", fontsize=12)
    plt.ylabel("Lifespan Standard Deviation", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig2_optimization_curve.png"), dpi=300)
    plt.close()

    # Fig 3: 3D Surface
    fig = plt.figure(figsize=(12, 8))
    ax3d = fig.add_subplot(111, projection='3d')
    LAM, SIG = np.meshgrid(lambda_vals, sigma_AB_vals)
    surf = ax3d.plot_surface(LAM, SIG, std_matrix, cmap='viridis_r', edgecolor='none', alpha=0.9)
    ax3d.set_title("3D Boundary of Robust Homogenität", fontsize=14)
    ax3d.set_xlabel(r"$\lambda$ (Shock Freq)", fontsize=12)
    ax3d.set_ylabel(r"$\sigma_{AB}$ (Randomization)", fontsize=12)
    ax3d.set_zlabel("Lifespan Std Dev", fontsize=12)
    fig.colorbar(surf, ax=ax3d, shrink=0.5, aspect=10)
    plt.savefig(os.path.join(output_dir, "fig3_robust_surface.png"), dpi=300)
    plt.close()

    print("Synthesis complete. Graphs generated.")

    # ==========================================
    # 4. Zip and Download
    # ==========================================
    zip_filename = "robust_threshold_synthesis.zip"
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files_in_dir in os.walk(output_dir):
            for file in files_in_dir:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.basename(file_path))

    files.download(zip_filename)

if __name__ == "__main__":
    explore_effective_thresholds()

Running Synthesis Phase Space Search: 500 grids...
Synthesis complete. Graphs generated.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>